# recursive_opt — Use-Case Experiment Suite (3 experiments each)

This notebook runs **3 complementary experiments per use case** to maximize the chance
of a successful / informative result, measures them in a comparison table, and
displays the winning artifact/code so you can inspect and reuse it.

## Read this first: what "offline" really means here
A Trace **optimizer** (OptoPrimeV2) calls an LLM — so *genuine recursive optimization
requires an API key* (`LIVE = True` below). What runs **without** a key is:
- the **evaluator / plumbing** (scoring a candidate is deterministic for the code surface),
- an **offline plumbing pre-flight** that installs a *hand-written* improved candidate to
  prove the score is climbable and the evaluator works — it does **not** discover the
  improvement, it only validates the surface.

So: **offline = validate the surface & evaluator. live = actually optimize.**
Every experiment below declares which mode it needs. Set `LIVE=True` + a key for the
real thing; leave `LIVE=False` to dry-run the plumbing and inspect the specs.

In [1]:
# ============================ CONFIGURATION ===============================
# LIVE=True runs REAL recursive optimization. The model is configured here so
# every optimizer / Trace-Bench inference path uses the same backend.
import os, sys, json, time, statistics, textwrap, math
from pathlib import Path

# Make Run-All robust from repo root, examples/, or nbconvert kernels whose cwd
# is not automatically inserted on sys.path. This is notebook-only path setup; it
# does not change the installed package.
_REPO_ROOT = Path.cwd()
if not (_REPO_ROOT / "opto").exists() and (_REPO_ROOT.parent / "opto").exists():
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

LIVE = True
MODEL = os.environ.get("RECURSIVE_OPT_MODEL") or os.environ.get("TRACE_LITELLM_MODEL") or "gpt-5.4-nano"
os.environ["RECURSIVE_OPT_MODEL"] = MODEL
os.environ["TRACE_LITELLM_MODEL"] = MODEL

# Budget - these map 1:1 to spec["budget"] (RecursiveOptBudget). Tune per run.
WALL_TIME_S          = 1800  # hard wall-clock cap per run (~30 min); the simplest guard
RUN_ITERATIONS       = 2     # optimizer update steps per seed/run
NUM_CANDIDATES       = 2     # proposals per optimizer step
MAX_OPTIMIZER_CALLS  = 8     # caps LLM proposal cost per seed/run
MAX_EVAL_CALLS       = 48    # standard cap for prompt/config/code runs
CAPABILITY_EVAL_CALLS = 96  # UC3 does train + final eval over 8 examples; 48 exhausted before save_priors
MAX_CANDIDATES       = 8     # caps search breadth (iterations * num_candidates)
os.environ["RECURSIVE_OPT_ITERATIONS"] = str(RUN_ITERATIONS)
os.environ["RECURSIVE_OPT_NUM_CANDIDATES"] = str(NUM_CANDIDATES)

# Task-eval bounds dominate cost more than anything. 8 examples is the current
# default here because earlier 4-example runs saturated too easily and were noisy.
MAX_EXAMPLES = 8
INNER_STEPS  = 0             # 0 = artifact-only (fast); >0 = real inner training (costly)
TIMEOUT_S    = 35            # per-eval timeout
os.environ["RECURSIVE_OPT_CAPABILITY_MAX_EXAMPLES"] = str(MAX_EXAMPLES)

# Two seeds are the minimum useful repeated live check; add seed 2 for publication runs.
SEEDS = [0, 1]

# Keep every generated memory/artifact folder under one run directory.
# If the notebook is launched from the repo root, this resolves to
# examples/notebook_outputs/...; if launched from examples/, it resolves to
# notebook_outputs/... under examples/. Override with RECURSIVE_OPT_OUTPUT_ROOT.
RUN_ID = os.environ.get("RECURSIVE_OPT_RUN_ID") or time.strftime("use_cases_%Y%m%d_%H%M%S")
_DEFAULT_OUTPUT_ROOT = (Path("examples/notebook_outputs/recursive_opt_use_cases")
                        if Path("examples").exists()
                        else Path("notebook_outputs/recursive_opt_use_cases"))
OUTPUT_ROOT = Path(os.environ.get("RECURSIVE_OPT_OUTPUT_ROOT", str(_DEFAULT_OUTPUT_ROOT))) / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# credit_horizon controls how much per-example guide feedback the optimizer sees.
# For UC6 we fix it at the previously best setting ("step") and compare trace designs.

# NOTE on the hf:gsm8k alias: recursive_opt currently redirects hf:gsm8k ->
# internal:multiobjective_gsm8k. We use internal:* families directly to avoid ambiguity.
if LIVE:
    from opto.features.recursive_opt.runmode import preflight_model
    from opto.features.recursive_opt.tracebench import ensure_default_task_adapter
    preflight_model(MODEL)
    ensure_default_task_adapter(require=True)
print("LIVE =", LIVE, "| model =", MODEL,
      "| iterations =", RUN_ITERATIONS, "| candidates =", NUM_CANDIDATES,
      "| examples =", MAX_EXAMPLES, "| seeds =", SEEDS,
      "| eval_calls =", MAX_EVAL_CALLS, "| capability_eval_calls =", CAPABILITY_EVAL_CALLS,
      "| output_root =", OUTPUT_ROOT)


LIVE = True | model = gpt-5.4-nano | iterations = 2 | candidates = 2 | examples = 8 | seeds = [0, 1] | eval_calls = 48 | capability_eval_calls = 96 | output_root = notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157


In [2]:
# ============================ DRY HARNESS ================================
# One place that runs a spec, collects initial/final scores and artifact refs
# across seeds, and renders comparison tables. Every use case reuses this.
from opto.features.recursive_opt import run_spec, make_level_spec, MemoryLite
from opto.features.recursive_opt.budget import RecursiveOptBudget, reset_budget


def budget_block():
    """Standard budget dict from the config knobs above (spec['budget'] keys)."""
    return {"wall_time_s": WALL_TIME_S, "optimizer_llm_calls": MAX_OPTIMIZER_CALLS,
            "eval_llm_calls": MAX_EVAL_CALLS, "candidates": MAX_CANDIDATES,
            "on_exceed": "return_best"}


def make_budget():
    """Finite budget for direct optimize() calls outside run_spec."""
    return RecursiveOptBudget(
        max_wall_time_s=WALL_TIME_S,
        max_optimizer_llm_calls=MAX_OPTIMIZER_CALLS,
        max_eval_llm_calls=MAX_EVAL_CALLS,
        max_candidates=MAX_CANDIDATES,
        stop_policy="return_best",
    )


def reset_standard_budget():
    """Reset direct optimize() calls to the same envelope as spec runs."""
    reset_budget(make_budget())


def memory_path(name):
    """Return an experiment memory path under the common OUTPUT_ROOT."""
    safe = str(name).strip().strip("./") or "mem"
    return str(OUTPUT_ROOT / safe)


def tracebench_block():
    """Standard real-adapter bounds (spec['tracebench'] keys)."""
    return {"max_examples": MAX_EXAMPLES, "inner_steps": INNER_STEPS,
            "timeout_seconds": TIMEOUT_S}


def _one_line_error(exc):
    """Compact, table-safe error summary using the first non-empty message line."""
    lines = [line.strip() for line in str(exc).splitlines() if line.strip()]
    detail = lines[0][:160] if lines else repr(exc)[:160]
    return f"{type(exc).__name__}: {detail}"


def _finite(values):
    """Return finite float values only."""
    out = []
    for value in values:
        try:
            f = float(value)
        except (TypeError, ValueError):
            continue
        if math.isfinite(f):
            out.append(f)
    return out


def _fmt(value):
    """Format optional numeric values for markdown tables."""
    if value is None:
        return "-"
    try:
        f = float(value)
    except (TypeError, ValueError):
        return str(value)
    return f"{f:.3f}" if math.isfinite(f) else "-"


def _artifact_file(root):
    """Return the JSONL file where reusable artifacts are persisted."""
    return str(Path(root) / "artifacts.jsonl")


def _artifact_ref(root, artifact_id=None):
    """Reference the persisted artifact by file plus optional artifact id."""
    path = _artifact_file(root)
    return f"{path}#{artifact_id}" if artifact_id else path


def _result_mean(result):
    """Mean final score for a result dict, or None when no run succeeded."""
    scores = _finite(result.get("scores", []))
    return statistics.mean(scores) if scores else None


def initial_score_for_spec(spec, level_id=None, run_name=None):
    """Evaluate the unoptimized seed artifact once, using the same real adapter bounds."""
    if not LIVE:
        return None, None
    import opto.features.recursive_opt.spec as spec_mod
    try:
        reset_standard_budget()
        lid = level_id or spec["levels"][-1]["id"]
        base_root = Path(run_name or spec.get("memory_root", "mem")).name
        probe_spec = {**spec, "memory_root": memory_path(f"_initial_probes/{base_root}")}
        spec_mod.validate_spec(probe_spec)
        if "tracebench" in probe_spec:
            from opto.features.recursive_opt import tracebench as TB
            TB.configure_tracebench_adapter(probe_spec.get("tracebench") or {}, require=True)
        families = probe_spec.get("families", {})
        memory = MemoryLite(root=probe_spec["memory_root"])
        for level_spec in probe_spec["levels"]:
            level = spec_mod.compile_level(level_spec, memory, families, probe_spec.get("scoring"))
            if level_spec["id"] == lid:
                score, _data = spec_mod._final_eval(level, level_spec, families)
                score = spec_mod._clamp(score, spec_mod._clip_bounds(probe_spec.get("scoring")))
                return float(score), None
        return None, f"level {lid!r} not found"
    except Exception as exc:
        return None, _one_line_error(exc)


def run_spec_seeds(spec, seeds=SEEDS, level_id=None, run_name=None):
    """Run a spec across seeds and keep failures visible without stopping the suite."""
    scores, walls, artifact, aid, errors = [], [], None, None, []
    artifact_ref, best_score = None, None
    lid = level_id or spec["levels"][-1]["id"]
    base_root = Path(run_name or spec.get("memory_root", "mem")).name
    if not LIVE:
        from opto.features.recursive_opt import validate_spec
        validate_spec(spec)
        return {"scores": [], "initial": None, "wall_s": None,
                "artifact": "(dry-run: set LIVE=True to optimize)",
                "artifact_id": None, "artifact_file": None, "dry": True}

    initial, initial_error = initial_score_for_spec(spec, level_id=lid, run_name=base_root)
    if initial_error:
        errors.append(f"initial: {initial_error}")
    for seed in seeds:
        try:
            reset_standard_budget()
            root = memory_path(f"{base_root}_{seed}")
            out = run_spec({**spec, "memory_root": root})
            r = out["results"][lid]
            score = float(r["score"])
            scores.append(score); walls.append(float(r["wall_s"]))
            ref = _artifact_ref(root, r.get("artifact_id"))
            if best_score is None or score > best_score:
                best_score = score
                artifact, aid, artifact_ref = r["artifact"], r.get("artifact_id"), ref
        except Exception as exc:
            errors.append(f"seed {seed}: {_one_line_error(exc)}")
    return {"scores": scores, "initial": initial,
            "wall_s": round(statistics.mean(walls), 1) if walls else None,
            "artifact": artifact or "(no successful seed)", "artifact_id": aid,
            "artifact_file": artifact_ref, "errors": errors, "dry": False}


def summarize(rows):
    """rows: list of (label, result_dict). Returns a markdown comparison table."""
    head = ("| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |\n"
            "|---|---:|---:|---:|---:|---:|---:|---|---|")
    lines = [head]
    for label, r in rows:
        if r.get("dry"):
            lines.append(f"| {label} | - | dry-run | - | - | 0 | - | - | set LIVE=True |")
            continue
        scores = _finite(r.get("scores", []))
        mean = statistics.mean(scores) if scores else None
        std = statistics.pstdev(scores) if len(scores) > 1 else None
        delta = (mean - r["initial"]) if mean is not None and r.get("initial") is not None else None
        notes = "; ".join(r.get("errors", [])[:2])
        if len(r.get("errors", [])) > 2:
            notes += f"; +{len(r['errors'])-2} more"
        artifact_file = r.get("artifact_file") or "-"
        lines.append(f"| {label} | {_fmt(r.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                     f"{_fmt(std)} | {len(scores)} | {_fmt(r.get('wall_s'))} | `{artifact_file}` | {notes} |")
    return "\n".join(lines)


def best_of(rows):
    """Return (label, result) with the highest mean score (ties -> lower wall_s)."""
    scored = [(l, r) for l, r in rows if _result_mean(r) is not None]
    if not scored:
        return None
    return max(scored, key=lambda lr: (_result_mean(lr[1]), -(lr[1].get("wall_s") or 1e9)))


from IPython.display import Markdown, display

def show_table(title, rows):
    display(Markdown(f"### {title}\n" + summarize(rows)))
    b = best_of(rows)
    if b:
        display(Markdown(f"**Best: `{b[0]}`** — artifact file: `{b[1].get('artifact_file') or '-'}`"))
        print(textwrap.shorten(str(b[1]["artifact"]), 1200, placeholder=" ...[truncated]"))


def _read_jsonl(path):
    """Read a JSONL file defensively for cross-run summaries."""
    p = Path(path)
    if not p.exists():
        return []
    rows = []
    for line in p.read_text().splitlines():
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def _best_artifact_from_dir(mem_dir):
    """Best finite artifact record in a MemoryLite directory."""
    records = _read_jsonl(Path(mem_dir) / "artifacts.jsonl")
    valid = [r for r in records if _finite([r.get("score")])]
    return max(valid, key=lambda r: float(r["score"])) if valid else None


def _initial_from_dir(mem_dir):
    """Best-effort initial score from the first persisted artifact or episode."""
    for filename in ("artifacts.jsonl", "episodes.jsonl"):
        records = _read_jsonl(Path(mem_dir) / filename)
        for record in records:
            scores = _finite([record.get("score")])
            if scores:
                return scores[0]
    return None


UC_DIR_PREFIXES = {
    "UC1 component code": "mem_uc1",
    "UC2 setup/config": "mem_uc2",
    "UC3 capability": "mem_uc3",
    "UC4 family/transfer": "mem_uc4",
    "UC5 optimizer/tool": "mem_uc5",
    "UC6 trace feedback": "mem_uc6",
}


def summarize_past_runs(base_dir=None):
    """Scan previous notebook output folders and summarize persisted artifact scores."""
    base = Path(base_dir or OUTPUT_ROOT.parent)
    rows = []
    for run_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for uc_name, prefix in UC_DIR_PREFIXES.items():
            mem_dirs = sorted(p for p in run_dir.glob(f"{prefix}*") if p.is_dir())
            if not mem_dirs:
                continue
            best, best_dir = None, None
            finals, initials = [], []
            for mem_dir in mem_dirs:
                init = _initial_from_dir(mem_dir)
                if init is not None:
                    initials.append(init)
                art = _best_artifact_from_dir(mem_dir)
                if art is None:
                    continue
                score = float(art["score"])
                finals.append(score)
                if best is None or score > float(best["score"]):
                    best, best_dir = art, mem_dir
            if best is None:
                continue
            rows.append({
                "run": run_dir.name,
                "use_case": uc_name,
                "initial_mean": statistics.mean(initials) if initials else None,
                "best_score": float(best["score"]),
                "final_mean": statistics.mean(finals) if finals else None,
                "n_dirs": len(mem_dirs),
                "artifact_file": _artifact_ref(best_dir, best.get("artifact_id")),
            })
    return rows


def past_runs_table(rows):
    """Render the cross-run artifact summary."""
    head = "| run | use case | initial mean | final mean | best score | n dirs | best artifact file |\n|---|---|---:|---:|---:|---:|---|"
    lines = [head]
    for row in rows:
        lines.append(f"| {row['run']} | {row['use_case']} | {_fmt(row['initial_mean'])} | "
                     f"{_fmt(row['final_mean'])} | {_fmt(row['best_score'])} | {row['n_dirs']} | "
                     f"`{row['artifact_file']}` |")
    return "\n".join(lines)

print("harness ready")


harness ready


---
### Root-cause diagnostics
These quick checks separate optimizer behavior from benchmark shape. They are intentionally small: probe score spread tells us whether a surface has enough room to learn, while code-baseline probes show when UC1/UC5 are saturated by a narrow deterministic validator rather than by broad benchmark performance. BBEH is probed here for task-shape awareness, but it is not used in UC3 because its Trace-Bench bundle is raw/code-artifact style rather than a prompt-capability surface.

In [3]:
from opto.features.recursive_opt.spec import score_spread
from opto.features.recursive_opt.tracebench import make_code_evaluator, configure_tracebench_adapter

if LIVE:
    configure_tracebench_adapter(tracebench_block(), require=True)
    diagnostic_rows = []
    probe_prompts = [
        {},
        {"starting_artifact": "Answer directly."},
        {"starting_artifact": "Plan step by step, then verify the answer before replying."},
    ]
    for task in ["internal:multiobjective_gsm8k", "internal:multiobjective_bbeh"]:
        try:
            spread = score_spread(task, probes=probe_prompts)
            scores = [r.get("score") for r in spread["rows"]]
            diagnostic_rows.append((task, spread["valid_spread"], spread["invalid_probes"], scores))
        except Exception as exc:
            diagnostic_rows.append((task, None, None, _one_line_error(exc)))

    code_probe = make_code_evaluator("internal:batch_design", "batch_design")
    code_rows = []
    for label, fn in [
        ("take_first", lambda n, k: list(range(k))),
        ("take_last", lambda n, k: list(range(n-k, n))),
        ("stride", lambda n, k: list(range(0, n, max(1, n//k)))[:k]),
        ("hard_mod3", lambda n, k: [i for i in range(n) if i % 3 == 0][:k]),
    ]:
        score, feedback = code_probe(lambda **kw: fn(**kw), "internal:batch_design")
        code_rows.append((label, score, feedback))

    lines = ["| probe | spread/score | details |", "|---|---:|---|"]
    for task, spread, invalid, scores in diagnostic_rows:
        lines.append(f"| {task} score spread | {_fmt(spread)} | invalid={invalid}; scores={scores} |")
    for label, score, feedback in code_rows:
        lines.append(f"| batch_design baseline `{label}` | {_fmt(score)} | {feedback[:180]} |")
    display(Markdown("\n".join(lines)))
else:
    display(Markdown("Diagnostics skipped: set `LIVE=True`."))


/home/xav/miniconda3/envs/humanllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


| probe | spread/score | details |
|---|---:|---|
| internal:multiobjective_gsm8k score spread | 0.049 | invalid=0; scores=[-0.16187500000000002, -0.113375, -0.14450000000000002] |
| internal:multiobjective_bbeh score spread | 1.000 | invalid=0; scores=[0.999970254625, -6.702412500070309e-06, -1.4467537500140181e-05] |
| batch_design baseline `take_first` | 0.800 | [batch_design@internal:batch_design] validation_pool n=12, k=4; hard/failing indices are [0, 3, 6, 9] (defined by idx % 3 == 0); picked [0, 1, 2, 3]; hard_items=2/4; diversity=1.00 |
| batch_design baseline `take_last` | 0.700 | [batch_design@internal:batch_design] validation_pool n=12, k=4; hard/failing indices are [0, 3, 6, 9] (defined by idx % 3 == 0); picked [8, 9, 10, 11]; hard_items=1/4; diversity=1. |
| batch_design baseline `stride` | 1.000 | [batch_design@internal:batch_design] validation_pool n=12, k=4; hard/failing indices are [0, 3, 6, 9] (defined by idx % 3 == 0); picked [0, 3, 6, 9]; hard_items=4/4; diversity=1.00 |
| batch_design baseline `hard_mod3` | 1.000 | [batch_design@internal:batch_design] validation_pool n=12, k=4; hard/failing indices are [0, 3, 6, 9] (defined by idx % 3 == 0); picked [0, 3, 6, 9]; hard_items=4/4; diversity=1.00 |

---
## Use Case 1 — Optimize & validate a NEW Trace component (code surface) ⭐ strongest today

**Why:** the code surface has a deterministic evaluator, so the signal is clean and the
before/after is a real diff. Best for: a new trainer hot-path, batch sampler, trace
summarizer, validator, or small optimizer-tool helper.

**3 experiments** (complementary angles to maximize success):
1. **batch_design** on `internal:batch_design` — known-climbable (0.8→1.0 offline).
2. **trace_summarizer** on `internal:code_param` — multi-criteria evaluator (keep error evidence + be concise).
3. **same component, harder objective** — stress the optimizer with a stricter objective string.

**Mode:** offline pre-flight proves the surface; **set LIVE=True for the real rewrite.**

In [4]:
# Use Case 1 — code surface. Uses ComponentSpec + CodeArtifactLevel via optimize().
# Interpretation: this proves optimizer-to-code rewriting, validation, rollback, and
# artifact persistence. It is intentionally narrow; saturation means the validator is
# easy, not that the learned component generalizes to all training loops.
from opto.features.recursive_opt import CodeArtifactLevel, ComponentSpec, optimize, RecursiveGuide
from opto.features.recursive_opt.tracebench import make_code_evaluator, make_dataset
import opto.trace as trace


def run_code_experiment(name, task_id, objective, seeds=SEEDS, memory_name=None):
    """One code-surface experiment across isolated memory roots per seed."""
    if not LIVE:
        return {"scores": [], "initial": None, "wall_s": None,
                "artifact": f"(dry-run) component='{name}' task='{task_id}'; set LIVE=True to optimize",
                "artifact_id": None, "artifact_file": None, "dry": True}
    scores, initial_scores, walls, final_code, errors = [], [], [], None, []
    best_ref, best_score, best_code = None, None, None
    for seed in seeds:
        try:
            root_name = memory_name or f"mem_uc1_{name}"
            root = memory_path(f"{root_name}_{seed}")
            mem = MemoryLite(root=root)
            spec = ComponentSpec(name=name, baseline=_BASELINES[name],
                                 evaluate=make_code_evaluator(task_id, name), objective=objective)
            level = CodeArtifactLevel(spec, memory=mem)
            guide = RecursiveGuide()
            initial_scores.append(float(guide(task_id, level.forward(task_id), None)[0]))
            reset_standard_budget()
            t0 = time.time()
            optimize(level, make_dataset([task_id], repeats=MAX_EXAMPLES), guide=guide,
                     iterations=RUN_ITERATIONS, num_candidates=NUM_CANDIDATES)
            # Code surfaces persist every validated implementation. Report and
            # re-score the best saved artifact so the table points at the reusable
            # solution, even if the Trainer's active slot moved on.
            best = mem.best_artifact(str(task_id), "code")
            if best is not None and level.parameters():
                level.parameters()[0]._data = best.content
            walls.append(round(time.time() - t0, 1))
            score = float(guide(task_id, level.forward(task_id), None)[0])
            if best is not None and float(best.score) >= score:
                score, final_code = float(best.score), best.content
                ref = _artifact_ref(root, best.artifact_id)
            else:
                final_code = level.current_code()
                ref = _artifact_file(root)
            scores.append(score)
            if best_score is None or score > best_score:
                best_score, best_ref, best_code = score, ref, final_code
        except Exception as exc:
            errors.append(f"seed {seed}: {_one_line_error(exc)}")
    return {"scores": scores, "initial": statistics.mean(initial_scores) if initial_scores else None,
            "wall_s": round(statistics.mean(walls), 1) if walls else None,
            "artifact": best_code or final_code or "(no successful seed)", "artifact_id": None,
            "artifact_file": best_ref, "errors": errors, "dry": False}


# Baselines kept deliberately weak so there is headroom to climb.
def _weak_batch(self, n, k): return list(range(k))
def _trunc_summary(self, trace_text): return str(trace_text)[-500:]
_BASELINES = {"batch_design": _weak_batch, "trace_summarizer": _trunc_summary}

uc1 = [
  ("batch_design (failure-balanced)",
   run_code_experiment("batch_design", "internal:batch_design",
                       "Select the hard/failing items before easy ones; maximize validator score.")),
  ("trace_summarizer (default)",
   run_code_experiment("trace_summarizer", "internal:code_param",
                       "Preserve failing-assertion evidence while removing noise; be concise.",
                       memory_name="mem_uc1_trace_summarizer_default")),
  ("trace_summarizer (strict)",
   run_code_experiment("trace_summarizer", "internal:code_param",
                       "Keep ALL error evidence, drop everything else, target <60 chars.",
                       memory_name="mem_uc1_trace_summarizer_strict")),
]
show_table("Use Case 1 — component code optimization", uc1)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 3644.05it/s]

Evaluating agent:   0%|          | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|██████████| 8/8 [00:00<00:00, 3729.93it/s]

...

Evaluating agent: 100%|██████████| 8/8 [00:00<00:00, 16312.32it/s]

[Step 1] Test/test_score: 0.9166666666666667
[Step 1] Algo/Average train score: 0.8231382978723405
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 0.9166666666666667
[Step 1] Update/best_candidate_mean_score: 0.9166666666666667
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: 0.8962765957446809
[Step 1] Update/exploration_candidates_mean_score: 0.8962765957446809
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.0
[Step 1] Sample/mean_score: 0.8962765957446809
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/__code:5: def _trunc_summary(self, trace_text):
    s = str(trace_tex

### Use Case 1 — component code optimization
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| batch_design (failure-balanced) | 0.800 | 1.000 | 0.200 | 0.000 | 2 | 4.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:71116` |  |
| trace_summarizer (default) | 0.750 | 0.813 | 0.063 | 0.063 | 2 | 3.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_trace_summarizer_default_1/artifacts.jsonl#internal:code_param:code:1:82514` |  |
| trace_summarizer (strict) | 0.750 | 0.833 | 0.083 | 0.083 | 2 | 3.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_trace_summarizer_strict_1/artifacts.jsonl#internal:code_param:code:1:89689` |  |

**Best: `batch_design (failure-balanced)`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:71116`

def _weak_batch(self, n, k): hard = [i for i in range(n) if i % 3 == 0] rest = [i for i in range(n) if i % 3 != 0] return (hard + rest)[:k]


---
## Use Case 2 — Learn the best SETUP / default prompt for a family (config surface)

**Why:** optimize *which existing components & artifact* to use. Under `INNER_STEPS=0`
only **causally-active** fields move score: `starting_artifact`, `initial_knowledge`,
`trace_type`. (Trainer/batch only activate at `INNER_STEPS>0` — the contract enforces this.)

**3 experiments:**
1. **artifact menu** — search a small set of prompt strategies (incl. empty control arm).
2. **+ initial_knowledge** — add a knowledge-injection target.
3. **warm vs cold** — same spec with `reuse_priors` on, to measure prior transfer.

**Mode:** needs LIVE + Trace-Bench (real task scores).

In [5]:
# Use Case 2 — config surface via run_spec. Active fields only (INNER_STEPS=0).
# Use a prompt-compatible Trace-Bench task: starting_artifact is a system prompt
# here, not a numeric parameter. Scores are real adapter scores over MAX_EXAMPLES.
FAMILY_TASK = "internal:multiobjective_gsm8k"
ART_MENU = ["", "Answer directly.", "Plan step by step, then answer.",
            "Plan step by step, then verify the answer before replying."]


def config_spec(targets, reuse=False, extra_constraints=None, memory_root="./mem_uc2"):
    cons = {"starting_artifact": ART_MENU}
    if extra_constraints:
        cons.update(extra_constraints)
    return {"families": {"reasoning": [FAMILY_TASK]},
            "memory_root": memory_root, "reuse_priors": reuse,
            "budget": budget_block(), "tracebench": tracebench_block(),
            "levels": [ make_level_spec(
                id="o1_setup", surface="config", family="reasoning", task=FAMILY_TASK,
                targets=targets, constraints=cons,
                fixed={"optimizer": "OptoPrimeV2", "trace_type": "internal",
                       "credit_horizon": "step", "trainer": "PrioritySearch"},
                iterations=RUN_ITERATIONS)]}

uc2 = [
  ("artifact only",       run_spec_seeds(config_spec(["starting_artifact"], memory_root="./mem_uc2_artifact_only"),
                                         run_name="mem_uc2_artifact_only")),
  ("artifact+knowledge",  run_spec_seeds(config_spec(["starting_artifact", "initial_knowledge"],
                                                      memory_root="./mem_uc2_artifact_knowledge"),
                                         run_name="mem_uc2_artifact_knowledge")),
  ("artifact (warm prior)", run_spec_seeds(config_spec(["starting_artifact"], reuse=True,
                                                        memory_root="./mem_uc2_warm_prior"),
                                           run_name="mem_uc2_warm_prior")),
]
show_table("Use Case 2 — setup/config optimization", uc2)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:11<00:11, 11.66s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:11<00:00,  5.83s/it]

Evaluating agent:   0%|          | 0/2 [00:00<?, ?it/s]

...

Evaluating agent: 100%|██████████| 2/2 [00:11<00:00,  5.69s/it]

[Step 1] Test/test_score: -0.155625
[Step 1] Algo/Average train score: -0.157875
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: -0.151125
[Step 1] Update/best_candidate_mean_score: -0.151125
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: -0.15231250000000002
[Step 1] Update/exploration_candidates_mean_score: -0.15231250000000002
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.0
[Step 1] Sample/mean_score: -0.149625
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/level_config:8: starting_artifact: Plan step by step, then answer.


### Use Case 2 — setup/config optimization
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| artifact only | -0.160 | -0.144 | 0.016 | 0.003 | 2 | 57.100 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:38164` |  |
| artifact+knowledge | -0.169 | -0.153 | 0.016 | 0.003 | 2 | 60.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_knowledge_0/artifacts.jsonl#reasoning:config:0:21713` |  |
| artifact (warm prior) | -0.164 | -0.147 | 0.016 | 0.005 | 2 | 58.400 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_warm_prior_0/artifacts.jsonl#reasoning:config:0:72083` |  |

**Best: `artifact only`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:38164`

starting_artifact: Plan step by step, then verify the answer before replying.


---
## Use Case 3 — Discover a new CAPABILITY from a spec + objectives (capability surface)

**Why:** synthesize a capability artifact (a skill-like text) that satisfies a
natural-language spec while trading off objectives (accuracy↑, cost↓).

**3 experiments:** three different **seed specifications** for the same objective set, to
see which framing the optimizer can push furthest (a complementary-results search).

**Mode:** needs LIVE + Trace-Bench. Uses the `capability` surface in a spec.

In [6]:
# Use Case 3 — capability surface. Three seed framings; pareto over (accuracy, cost).
# Keep this on prompt-compatible GSM8K. BBEH is intentionally excluded here: the
# diagnostic probe showed it is a raw/code-artifact task for this adapter, so a
# natural-language capability prompt is evaluated as invalid code. That belongs to
# code-artifact experiments, not this prompt-capability surface.
from opto.features.recursive_opt.tracebench import make_multiobjective_evaluator

CAP_TASKS = ["internal:multiobjective_gsm8k"]
CAP_OBJECTIVES = {"accuracy": "max", "cost": "min"}
_cap_evaluator = make_multiobjective_evaluator(
    CAP_TASKS,
    CAP_OBJECTIVES,
    required_terms=("plan", "verify"),
)


def capability_spec(seed_text, memory_root="./mem_uc3"):
    cap_budget = {**budget_block(), "eval_llm_calls": CAPABILITY_EVAL_CALLS}
    return {"families": {"reasoning": CAP_TASKS}, "memory_root": memory_root,
            "budget": cap_budget, "tracebench": tracebench_block(),
            "levels": [ make_level_spec(
                id="cap", surface="capability", family="reasoning", task=CAP_TASKS[0],
                seed=seed_text, evaluator=_cap_evaluator,
                objective_config={"mode": "pareto", "minimize": ["cost"]},
                iterations=RUN_ITERATIONS)]}

uc3 = [
  ("seed: terse",   run_spec_seeds(capability_spec("Solve correctly using the fewest words.", "./mem_uc3_terse"),
                                   run_name="mem_uc3_terse")),
  ("seed: verify",  run_spec_seeds(capability_spec("Make a short plan; solve; then verify/check the answer before replying.", "./mem_uc3_verify"),
                                   run_name="mem_uc3_verify")),
  ("seed: decompose", run_spec_seeds(capability_spec("Plan, decompose into sub-steps, solve each, then verify before answering.", "./mem_uc3_decompose"),
                                     run_name="mem_uc3_decompose")),
]
show_table("Use Case 3 — capability discovery", uc3)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:09<00:09,  9.42s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:10<00:00,  4.40s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:10<00:00,  5.15s/it]

...

Evaluating agent: 100%|██████████| 2/2 [00:10<00:00,  5.27s/it]

[Step 1] Test/test_score: 1.4391666666666667
[Step 1] Algo/Average train score: 1.4391666666666667
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 1
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 2
[Step 1] Update/best_candidate_priority: 1.4391666666666667
[Step 1] Update/best_candidate_mean_score: 1.4391666666666667
[Step 1] Update/best_candidate_num_rollouts: 2
[Step 1] Update/num_exploration_candidates: 1
[Step 1] Update/exploration_candidates_mean_priority: 1.4391666666666667
[Step 1] Update/exploration_candidates_mean_score: 1.4391666666666667
[Step 1] Update/exploration_candidates_average_num_rollouts: 2.0
[Step 1] Sample/mean_score: 1.4391666666666667
[Step 1] Sample/num_samples: 1
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 3
[Step 1] Parameter/capability:8: Plan, decompose into sub-steps, solve each, then verify

### Use Case 3 — capability discovery
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| seed: terse | 0.968 | 0.905 | -0.062 | 0.062 | 2 | 37.100 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_terse_0/artifacts.jsonl#reasoning:capability:0:99191` |  |
| seed: verify | 1.441 | 1.441 | 0.000 | 0.000 | 2 | 63.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:31701` |  |
| seed: decompose | 1.439 | 1.439 | 0.000 | 0.000 | 2 | 47.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:78235` |  |

**Best: `seed: verify`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:31701`

Make a short plan; solve; then verify/check the answer before replying.


---
## Use Case 4 — Family policy (O2) & transferable prior (O3) — EXPERIMENTAL

**Why:** learn config per family (O2) and induce a prior validated on a held-out family (O3).
Both analyses agree this is *mechanically real but not yet a reliable win* — treat results
as exploratory and require warm>cold by >1σ on ≥2 families before believing transfer.

**3 experiments:**
1. **O2 only** — one family-policy level over 2 internal families.
2. **O2→O3 chain** — add a prior level with enforced `depends_on`.
3. **O3 warm vs cold** — re-run with `reuse_priors` to measure transfer.

**Mode:** needs LIVE. Keep families internal + small for the 30-min budget.

In [7]:
# Use Case 4 — O2/O3 via multi-level spec. Prompt-compatible internal
# families keep starting_artifact transfer meaningful and bounded. If both cold
# and warm O3 saturate, the conclusion is that this task mix is too easy for
# transfer evidence, not that transfer is universally solved.
FAMS = {"gsm8k": ["internal:multiobjective_gsm8k"], "bbeh": ["internal:multiobjective_bbeh"]}


def o2_spec(memory_root="./mem_uc4_o2"):
    return {"families": FAMS, "memory_root": memory_root,
            "budget": budget_block(), "tracebench": tracebench_block(),
            "levels": [ make_level_spec(id="o2", surface="family_policy",
                families=list(FAMS), targets=["starting_artifact"],
                fixed={"trace_type": "internal", "credit_horizon": "step"},
                iterations=RUN_ITERATIONS)]}


def o2o3_spec(reuse=False, memory_root="./mem_uc4_o3"):
    return {"families": FAMS, "memory_root": memory_root, "reuse_priors": reuse,
            "budget": budget_block(), "tracebench": tracebench_block(),
            "levels": [
              make_level_spec(id="o2", surface="family_policy", families=list(FAMS),
                  targets=["starting_artifact"], fixed={"trace_type":"internal", "credit_horizon":"step"},
                  iterations=RUN_ITERATIONS),
              make_level_spec(id="o3", surface="prior", families=list(FAMS),
                  targets=["starting_artifact"], fixed={"trace_type":"internal", "credit_horizon":"step"},
                  depends_on=["o2"], iterations=RUN_ITERATIONS)]}

uc4 = [
  ("O2 family policy",      run_spec_seeds(o2_spec("./mem_uc4_o2_policy"), level_id="o2", run_name="mem_uc4_o2_policy")),
  ("O2->O3 (cold)",         run_spec_seeds(o2o3_spec(reuse=False, memory_root="./mem_uc4_o3_cold"),
                                           level_id="o3", run_name="mem_uc4_o3_cold")),
  ("O2->O3 (warm prior)",   run_spec_seeds(o2o3_spec(reuse=True, memory_root="./mem_uc4_o3_warm"),
                                           level_id="o3", run_name="mem_uc4_o3_warm")),
]
show_table("Use Case 4 — family policy & transfer (experimental)", uc4)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:10<00:10, 10.23s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:10<00:00,  4.33s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:10<00:00,  5.22s/it]

...

Evaluating agent: 100%|██████████| 2/2 [00:00<00:00, 728.94it/s]

[Step 1] Test/test_score: 0.9999852669437501
[Step 1] Algo/Average train score: 0.9999850017624998
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 0.999991285075
[Step 1] Update/best_candidate_mean_score: 0.999991285075
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: 0.999989322109375
[Step 1] Update/exploration_candidates_mean_score: 0.999989322109375
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.5
[Step 1] Sample/mean_score: 0.9999826443812498
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/transfer_prior:5: starting_artifact: 


### Use Case 4 — family policy & transfer (experimental)
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| O2 family policy | 0.420 | 0.418 | -0.002 | 0.000 | 2 | 50.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o2_policy_0/artifacts.jsonl#*:family_policy:0:8942` |  |
| O2->O3 (cold) | 1.000 | 1.000 | -0.000 | 0.000 | 2 | 3.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_cold_1/artifacts.jsonl#*:prior:0:83127` |  |
| O2->O3 (warm prior) | 1.000 | 1.000 | -0.000 | 0.000 | 2 | 3.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_warm_1/artifacts.jsonl#*:prior:0:12148` |  |

**Best: `O2->O3 (warm prior)`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_warm_1/artifacts.jsonl#*:prior:0:12148`

starting_artifact:


---
## Use Case 5 — Optimize an optimizer TOOL (model it as code) — EXPERIMENTAL

**Why:** both analyses recommend optimizing a tool by modeling it as a `CodeArtifactLevel`
component with a deterministic evaluator — the same proven machinery as Use Case 1, applied
to a tool's hot-path (e.g. a failure-retrieval ranker, a subset selector).

**3 experiments:** three tool baselines of increasing weakness, to see how much the
optimizer can recover — a complementary difficulty sweep.

**Mode:** offline pre-flight + LIVE for the real rewrite (reuses Use Case 1 machinery).

In [8]:
# Use Case 5 — two meanings of "optimizer tool".
# 5a is CODE-SURFACE optimization of a helper/selector function. The LLM rewrites
# component code and the reusable solution is saved as kind="code" in artifacts.jsonl.
# 5b is real optimizer-side tool calling: AgenticOptimizer calls registered helper
# tools (e.g. note, trace_search) before proposing an update and injects their
# evidence into optimizer feedback. It does NOT learn tool code or a tool list here.
def _baseline_take_first(self, n, k): return list(range(k))
def _baseline_take_last(self, n, k):  return list(range(n-k, n))
def _baseline_stride(self, n, k):     return list(range(0, n, max(1, n//k)))[:k]

uc5 = []
for key, label, fn in [("take_first", "code helper: take_first", _baseline_take_first),
                       ("take_last",  "code helper: take_last",  _baseline_take_last),
                       ("stride",     "code helper: stride",     _baseline_stride)]:
    _BASELINES["batch_design"] = fn
    uc5.append((label, run_code_experiment("batch_design", "internal:batch_design",
                "Select hard/failing items first to maximize validator score.",
                memory_name=f"mem_uc5_code_{key}")))
_BASELINES["batch_design"] = _weak_batch


def agentic_tool_spec(tools, label):
    spec = config_spec(
        ["starting_artifact"],
        memory_root=f"./mem_uc5_agentic_{label}",
        extra_constraints={"starting_artifact": ART_MENU},
    )
    spec["levels"] = [ make_level_spec(
        id=f"o1_agentic_{label}", surface="config", family="reasoning", task=FAMILY_TASK,
        targets=["starting_artifact"], constraints={"starting_artifact": ART_MENU},
        fixed={"optimizer": "OptoPrimeV2", "trace_type": "internal",
               "credit_horizon": "step", "trainer": "PrioritySearch"},
        agentic={"tool_budget": max(1, len(tools))}, tools=tools,
        iterations=RUN_ITERATIONS)]
    return spec

uc5 += [
    ("optimizer tools: note", run_spec_seeds(agentic_tool_spec(["note"], "note"),
                                             level_id="o1_agentic_note", run_name="mem_uc5_agentic_note")),
    ("optimizer tools: trace_search+note", run_spec_seeds(agentic_tool_spec(["trace_search", "note"], "trace_note"),
                                                          level_id="o1_agentic_trace_note", run_name="mem_uc5_agentic_trace_note")),
]
show_table("Use Case 5 — helper-code vs optimizer-side tool calling", uc5)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 2067.69it/s]

Evaluating agent:   0%|          | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|██████████| 8/8 [00:00<00:00, 4009.85it/s]

...

Evaluating agent: 100%|██████████| 2/2 [00:09<00:00,  4.89s/it]

[Step 1] Test/test_score: -0.1611875
[Step 1] Algo/Average train score: -0.16216666666666668
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 1
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 2
[Step 1] Update/best_candidate_priority: -0.1599375
[Step 1] Update/best_candidate_mean_score: -0.1599375
[Step 1] Update/best_candidate_num_rollouts: 2
[Step 1] Update/num_exploration_candidates: 1
[Step 1] Update/exploration_candidates_mean_priority: -0.1599375
[Step 1] Update/exploration_candidates_mean_score: -0.1599375
[Step 1] Update/exploration_candidates_average_num_rollouts: 2.0
[Step 1] Sample/mean_score: -0.166625
[Step 1] Sample/num_samples: 1
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 3
[Step 1] Parameter/level_config:14: starting_artifact: 


### Use Case 5 — helper-code vs optimizer-side tool calling
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| code helper: take_first | 0.800 | 1.000 | 0.200 | 0.000 | 2 | 3.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_take_first_0/artifacts.jsonl#internal:batch_design:code:1:15406` |  |
| code helper: take_last | 0.700 | 1.000 | 0.300 | 0.000 | 2 | 3.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:21749` |  |
| code helper: stride | 1.000 | 1.000 | 0.000 | 0.000 | 2 | 1.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:25130` |  |
| optimizer tools: note | -0.161 | -0.164 | -0.003 | 0.000 | 2 | 43.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_agentic_note_0/artifacts.jsonl#reasoning:config:0:93443` |  |
| optimizer tools: trace_search+note | -0.168 | -0.162 | 0.006 | 0.002 | 2 | 42.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_agentic_trace_note_1/artifacts.jsonl#reasoning:config:0:64119` |  |

**Best: `code helper: stride`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:25130`

def _baseline_stride(self, n, k): return list(range(0, n, max(1, n//k)))[:k]


---
## Use Case 6 — Which FEEDBACK CHANNEL helps the optimizer? (trace_type × credit_horizon) — EXPERIMENTAL

**Why:** two knobs shape the *evidence the optimizer sees* (both FEEDBACK-plumbed, not
score-plumbed — their benefit shows via better proposals over live steps, not a direct bonus):
- **`trace_type`** — `internal` / `otel` / `hybrid` (which trace backend feeds the optimizer).
- **`credit_horizon`** (NEW, from the credit-horizon patch) — how much per-example guide
  feedback is summarized for the meta-optimizer: `step` (per-step, ≤5), `episode` (top-3,
  default), `truncated` (1), `full` (all). Now an **active** target the contract accepts.

**3 experiments:**
1. **trace_type sweep** — internal vs otel vs hybrid (same artifact search).
2. **credit_horizon sweep** — step vs episode vs full (the newly-active knob).
3. **joint** — search both `trace_type` and `credit_horizon` together.

**Mode:** needs LIVE. **Requires the credit_horizon patch** for experiments 2-3 (without it
the contract marks `credit_horizon` inactive and will reject it as a target — which is itself
a useful signal that the patch isn't applied).

> **Best live testbed for credit_horizon:** `hf:bbeh_horizon` (long reasoning chains). With the patch, that family **auto-expands** into its subtasks, so step-vs-episode feedback has real headroom. Swap `FAMILY_TASK = "hf:bbeh_horizon"` once Trace-Bench has that dataset + a key is set.

In [9]:
# Use Case 6 — feedback channels. Hold credit_horizon at the strongest prior
# setting ("step") and focus on trace_type/design. This removes the noisy joint
# grid and asks one controlled question: which trace representation helps proposals?
def feedback_spec(level_id, trace_type):
    return {"families": {"reasoning": [FAMILY_TASK]}, "memory_root": f"./mem_uc6_{level_id}",
            "budget": budget_block(), "tracebench": tracebench_block(),
            "levels": [ make_level_spec(
                id=level_id, surface="config", family="reasoning", task=FAMILY_TASK,
                targets=["starting_artifact"], constraints={"starting_artifact": ART_MENU},
                fixed={"optimizer": "OptoPrimeV2", "trainer": "PrioritySearch",
                       "trace_type": trace_type, "credit_horizon": "step"},
                iterations=RUN_ITERATIONS)]}

uc6 = [(f"trace_type={tt} | credit_horizon=step",
        run_spec_seeds(feedback_spec(f"o1_trace_{tt}", tt), level_id=f"o1_trace_{tt}",
                       run_name=f"mem_uc6_trace_{tt}"))
       for tt in ["internal", "otel", "hybrid"]]

show_table("Use Case 6 — trace representation with fixed step credit", uc6)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:10<00:10, 10.36s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:11<00:00,  5.17s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:11<00:00,  5.95s/it]

...

Evaluating agent: 100%|██████████| 2/2 [00:11<00:00,  5.65s/it]

[Step 1] Test/test_score: -0.1459375
[Step 1] Algo/Average train score: -0.15740625000000003
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: -0.141875
[Step 1] Update/best_candidate_mean_score: -0.141875
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: -0.14903125
[Step 1] Update/exploration_candidates_mean_score: -0.14903125
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.5
[Step 1] Sample/mean_score: -0.15862500000000002
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/level_config:23: starting_artifact: Plan step by step, then verify the answer before replying.


### Use Case 6 — trace representation with fixed step credit
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| trace_type=internal | credit_horizon=step | -0.165 | -0.128 | 0.037 | 0.009 | 2 | 55.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:3911` |  |
| trace_type=otel | credit_horizon=step | -0.166 | -0.142 | 0.024 | 0.003 | 2 | 52.700 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:76288` |  |
| trace_type=hybrid | credit_horizon=step | -0.170 | -0.139 | 0.031 | 0.001 | 2 | 53.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_hybrid_1/artifacts.jsonl#reasoning:config:0:77200` |  |

**Best: `trace_type=internal | credit_horizon=step`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:3911`

starting_artifact: Answer directly.


---
## Master summary — all use cases at a glance

Run after the experiments above. Collects each use case's **best experiment** into one table
so you can see, for this budget, where recursive_opt delivered signal today.

In [10]:
# Master roll-up: all experiments, best per use case, and previous-run artifact summaries.
ALL = {"UC1 component code": uc1, "UC2 setup/config": uc2, "UC3 capability": uc3,
       "UC4 family/transfer": uc4, "UC5 optimizer/tool": uc5, "UC6 trace feedback": uc6}

flat = ["| use case | experiment | initial | mean score | delta | std | n | wall_s | best artifact file | best? |",
        "|---|---|---:|---:|---:|---:|---:|---:|---|---|"]
for uc, data in ALL.items():
    best = best_of(data)
    best_label = best[0] if best else None
    for label, result in data:
        scores = _finite(result.get("scores", []))
        mean = statistics.mean(scores) if scores else None
        std = statistics.pstdev(scores) if len(scores) > 1 else None
        delta = (mean - result["initial"]) if mean is not None and result.get("initial") is not None else None
        flat.append(f"| {uc} | {label} | {_fmt(result.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                    f"{_fmt(std)} | {len(scores)} | {_fmt(result.get('wall_s'))} | "
                    f"`{result.get('artifact_file') or '-'}` | {'yes' if label == best_label else ''} |")

display(Markdown("### All current-run results\n" + "\n".join(flat)))

best_rows = ["| use case | best experiment | initial | mean score | delta | n | wall_s | best artifact file |",
             "|---|---|---:|---:|---:|---:|---:|---|"]
for uc, data in ALL.items():
    b = best_of(data)
    if b is None:
        best_rows.append(f"| {uc} | (dry-run / no live result) | - | - | - | 0 | - | - |")
    else:
        label, result = b
        mean = _result_mean(result)
        delta = (mean - result["initial"]) if mean is not None and result.get("initial") is not None else None
        best_rows.append(f"| {uc} | {label} | {_fmt(result.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                         f"{len(_finite(result.get('scores', [])))} | {_fmt(result.get('wall_s'))} | "
                         f"`{result.get('artifact_file') or '-'}` |")
display(Markdown("### Best result per use case\n" + "\n".join(best_rows)))

past = summarize_past_runs(OUTPUT_ROOT.parent)
if past:
    display(Markdown("### Historical persisted-artifact summary\n" + past_runs_table(past)))
else:
    display(Markdown("### Historical persisted-artifact summary\nNo prior output folders found."))

display(Markdown("**Interpretation guardrails:** UC1/UC5 code-helper scores are validator evidence and can saturate; "
                 "UC2/UC6 are real Trace-Bench prompt/config scores over the configured examples; "
                 "UC3 now uses mixed prompt-compatible tasks; UC4 transfer is only meaningful when warm beats cold by more than run noise."))


### All current-run results
| use case | experiment | initial | mean score | delta | std | n | wall_s | best artifact file | best? |
|---|---|---:|---:|---:|---:|---:|---:|---|---|
| UC1 component code | batch_design (failure-balanced) | 0.800 | 1.000 | 0.200 | 0.000 | 2 | 4.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:71116` | yes |
| UC1 component code | trace_summarizer (default) | 0.750 | 0.813 | 0.063 | 0.063 | 2 | 3.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_trace_summarizer_default_1/artifacts.jsonl#internal:code_param:code:1:82514` |  |
| UC1 component code | trace_summarizer (strict) | 0.750 | 0.833 | 0.083 | 0.083 | 2 | 3.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_trace_summarizer_strict_1/artifacts.jsonl#internal:code_param:code:1:89689` |  |
| UC2 setup/config | artifact only | -0.160 | -0.144 | 0.016 | 0.003 | 2 | 57.100 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:38164` | yes |
| UC2 setup/config | artifact+knowledge | -0.169 | -0.153 | 0.016 | 0.003 | 2 | 60.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_knowledge_0/artifacts.jsonl#reasoning:config:0:21713` |  |
| UC2 setup/config | artifact (warm prior) | -0.164 | -0.147 | 0.016 | 0.005 | 2 | 58.400 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_warm_prior_0/artifacts.jsonl#reasoning:config:0:72083` |  |
| UC3 capability | seed: terse | 0.968 | 0.905 | -0.062 | 0.062 | 2 | 37.100 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_terse_0/artifacts.jsonl#reasoning:capability:0:99191` |  |
| UC3 capability | seed: verify | 1.441 | 1.441 | 0.000 | 0.000 | 2 | 63.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:31701` | yes |
| UC3 capability | seed: decompose | 1.439 | 1.439 | 0.000 | 0.000 | 2 | 47.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:78235` |  |
| UC4 family/transfer | O2 family policy | 0.420 | 0.418 | -0.002 | 0.000 | 2 | 50.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o2_policy_0/artifacts.jsonl#*:family_policy:0:8942` |  |
| UC4 family/transfer | O2->O3 (cold) | 1.000 | 1.000 | -0.000 | 0.000 | 2 | 3.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_cold_1/artifacts.jsonl#*:prior:0:83127` |  |
| UC4 family/transfer | O2->O3 (warm prior) | 1.000 | 1.000 | -0.000 | 0.000 | 2 | 3.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_warm_1/artifacts.jsonl#*:prior:0:12148` | yes |
| UC5 optimizer/tool | code helper: take_first | 0.800 | 1.000 | 0.200 | 0.000 | 2 | 3.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_take_first_0/artifacts.jsonl#internal:batch_design:code:1:15406` |  |
| UC5 optimizer/tool | code helper: take_last | 0.700 | 1.000 | 0.300 | 0.000 | 2 | 3.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:21749` |  |
| UC5 optimizer/tool | code helper: stride | 1.000 | 1.000 | 0.000 | 0.000 | 2 | 1.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:25130` | yes |
| UC5 optimizer/tool | optimizer tools: note | -0.161 | -0.164 | -0.003 | 0.000 | 2 | 43.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_agentic_note_0/artifacts.jsonl#reasoning:config:0:93443` |  |
| UC5 optimizer/tool | optimizer tools: trace_search+note | -0.168 | -0.162 | 0.006 | 0.002 | 2 | 42.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_agentic_trace_note_1/artifacts.jsonl#reasoning:config:0:64119` |  |
| UC6 trace feedback | trace_type=internal | credit_horizon=step | -0.165 | -0.128 | 0.037 | 0.009 | 2 | 55.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:3911` | yes |
| UC6 trace feedback | trace_type=otel | credit_horizon=step | -0.166 | -0.142 | 0.024 | 0.003 | 2 | 52.700 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:76288` |  |
| UC6 trace feedback | trace_type=hybrid | credit_horizon=step | -0.170 | -0.139 | 0.031 | 0.001 | 2 | 53.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_hybrid_1/artifacts.jsonl#reasoning:config:0:77200` |  |

### Best result per use case
| use case | best experiment | initial | mean score | delta | n | wall_s | best artifact file |
|---|---|---:|---:|---:|---:|---:|---|
| UC1 component code | batch_design (failure-balanced) | 0.800 | 1.000 | 0.200 | 2 | 4.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:71116` |
| UC2 setup/config | artifact only | -0.160 | -0.144 | 0.016 | 2 | 57.100 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:38164` |
| UC3 capability | seed: verify | 1.441 | 1.441 | 0.000 | 2 | 63.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:31701` |
| UC4 family/transfer | O2->O3 (warm prior) | 1.000 | 1.000 | -0.000 | 2 | 3.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_warm_1/artifacts.jsonl#*:prior:0:12148` |
| UC5 optimizer/tool | code helper: stride | 1.000 | 1.000 | 0.000 | 2 | 1.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:25130` |
| UC6 trace feedback | trace_type=internal | credit_horizon=step | -0.165 | -0.128 | 0.037 | 2 | 55.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:3911` |

### Historical persisted-artifact summary
| run | use case | initial mean | final mean | best score | n dirs | best artifact file |
|---|---|---:|---:|---:|---:|---|
| use_cases_live_20260613_215505 | UC1 component code | 0.775 | 0.936 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:0:13999` |
| use_cases_live_20260613_215505 | UC3 capability | 0.964 | 0.964 | 0.968 | 3 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc3_0/artifacts.jsonl#reasoning:capability:0:94067` |
| use_cases_live_20260613_215505 | UC4 family/transfer | -0.579 | -0.366 | -0.153 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc4_o3_0/artifacts.jsonl#<holdout>:prior:0:93503` |
| use_cases_live_20260613_215505 | UC5 optimizer/tool | 0.833 | 1.000 | 1.000 | 9 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc5_stride_0/artifacts.jsonl#internal:batch_design:code:0:28880` |
| use_cases_live_deep_20260614_000827 | UC1 component code | 0.775 | 0.939 | 1.000 | 4 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:48142` |
| use_cases_live_deep_20260614_000827 | UC2 setup/config | -0.149 | -0.149 | -0.143 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:18120` |
| use_cases_live_deep_20260614_000827 | UC4 family/transfer | 0.419 | 0.807 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc4_o3_warm_0/artifacts.jsonl#<holdout>:prior:0:3687` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | 0.434 | 0.534 | 1.000 | 10 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:84588` |
| use_cases_live_deep_20260614_000827 | UC6 trace feedback | -0.150 | -0.150 | -0.144 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:61144` |
| use_cases_live_deep_budgeted_20260614_012157 | UC1 component code | 0.767 | 0.882 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:71116` |
| use_cases_live_deep_budgeted_20260614_012157 | UC2 setup/config | -0.148 | -0.148 | -0.140 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:38164` |
| use_cases_live_deep_budgeted_20260614_012157 | UC3 capability | 1.262 | 1.262 | 1.441 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:31701` |
| use_cases_live_deep_budgeted_20260614_012157 | UC4 family/transfer | 0.419 | 0.807 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_cold_0/artifacts.jsonl#<holdout>:prior:0:24095` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | 0.435 | 0.535 | 1.000 | 10 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:25130` |
| use_cases_live_deep_budgeted_20260614_012157 | UC6 trace feedback | -0.136 | -0.136 | -0.118 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:3911` |
| use_cases_live_deep_fixed_20260614_004207 | UC1 component code | 0.767 | 0.890 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:70343` |
| use_cases_live_deep_fixed_20260614_004207 | UC2 setup/config | -0.151 | -0.151 | -0.143 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:34862` |
| use_cases_live_deep_fixed_20260614_004207 | UC4 family/transfer | 0.419 | 0.807 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc4_o3_cold_0/artifacts.jsonl#<holdout>:prior:0:61920` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | 0.435 | 0.535 | 1.000 | 10 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:74840` |
| use_cases_live_deep_fixed_20260614_004207 | UC6 trace feedback | -0.139 | -0.139 | -0.116 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc6_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:46689` |
| use_cases_live_fixed_20260613_221437 | UC1 component code | 0.775 | 0.960 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:0:85257` |
| use_cases_live_fixed_20260613_221437 | UC2 setup/config | -0.143 | -0.132 | -0.116 | 3 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc2_1/artifacts.jsonl#reasoning:config:2:64082` |
| use_cases_live_fixed_20260613_221437 | UC3 capability | 0.966 | 0.966 | 0.968 | 3 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc3_0/artifacts.jsonl#reasoning:capability:0:31303` |
| use_cases_live_fixed_20260613_221437 | UC4 family/transfer | 0.421 | 0.711 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc4_o3_2/artifacts.jsonl#<holdout>:prior:0:31092` |
| use_cases_live_fixed_20260613_221437 | UC5 optimizer/tool | 0.833 | 1.000 | 1.000 | 9 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc5_stride_0/artifacts.jsonl#internal:batch_design:code:0:86973` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | -0.144 | -0.144 | -0.120 | 21 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_ch_step_0/artifacts.jsonl#reasoning:config:0:28025` |

**Interpretation guardrails:** UC1/UC5 code-helper scores are validator evidence and can saturate; UC2/UC6 are real Trace-Bench prompt/config scores over the configured examples; UC3 now uses mixed prompt-compatible tasks; UC4 transfer is only meaningful when warm beats cold by more than run noise.